## Cálculo de BIAS WER

In [21]:
from tools.wer_utils import calculate_wer_from_dataframe
import pandas as pd
import json
import jiwer

full_df = pd.read_csv('dataset_normalized.csv')

# Load keywords and ground truth
with open('keywords.json', 'r') as f:
    keywords_data = json.load(f)

with open('ground_truth.json', 'r') as f:
    ground_truth_list = json.load(f)
    # Convert list to dict for easier lookup
    ground_truth_map = {item['id']: item['text'] for item in ground_truth_list}

In [22]:
def get_keyword_stats(reference, hypothesis, target_keywords, val_to_key=None):
    """
    Calculates keyword error stats for a single pair of reference and hypothesis.
    
    Args:
        reference (str): The reference text.
        hypothesis (str): The hypothesis text.
        target_keywords (set/list): Collection of keyword values to check.
        val_to_key (dict, optional): Map from keyword value to keyword type. 
                                     If provided, returns a list of detailed stats per keyword occurrence.
    
    Returns:
        tuple: (keyword_errors, keyword_count) if val_to_key is None
        list: List of dicts with {'key_type': ..., 'error': ...} if val_to_key is provided
    """
    # Align reference and hypothesis
    # jiwer.process_words returns an alignment object with nested lists
    out = jiwer.process_words(reference, hypothesis)
    
    # Get the alignment for the first (and only) sentence
    # out.alignments is a list of lists of AlignmentChunk
    alignment = out.alignments[0]
    # out.references is a list of lists of strings
    ref_tokens = out.references[0]
    
    errors = 0
    count = 0
    detailed_stats = []
    
    # Iterate through alignments
    for chunk in alignment:
        # chunk type: 'equal', 'substitute', 'delete', 'insert'
        # We only care about reference words that are keywords
        if chunk.type == 'insert':
            continue
            
        # Check reference words in this chunk
        for ref_idx in range(chunk.ref_start_idx, chunk.ref_end_idx):
            # Use ref_tokens instead of out.references
            ref_word = ref_tokens[ref_idx]
            
            # Check against keywords without normalization
            if ref_word in target_keywords:
                count += 1
                is_error = 1 if chunk.type != 'equal' else 0
                if is_error:
                    errors += 1
                
                if val_to_key:
                    detailed_stats.append({
                        'key_type': val_to_key[ref_word],
                        'error': is_error,
                        'count': 1
                    })
                    
    if val_to_key:
        return detailed_stats
    return errors, count

def calculate_bias_wer(row):
    audio_id = str(row['audio'])
    
    # Get reference
    reference = ground_truth_map.get(audio_id, "")
    if not reference:
        return None
        
    hypothesis = str(row['text_normalized'])
    
    # Get keywords
    if audio_id not in keywords_data:
        return pd.Series([0.0, 0])
        
    # Extract keyword values without normalization
    # keywords_data[audio_id]['keywords'] is a list of dicts
    target_keywords = [k['val'] for k in keywords_data[audio_id]['keywords']]
    
    if not target_keywords:
        return pd.Series([0.0, 0])
        
    keyword_errors, keyword_count = get_keyword_stats(reference, hypothesis, target_keywords)
                    
    if keyword_count == 0:
        return pd.Series([0.0, 0])
        
    return pd.Series([keyword_errors / keyword_count, keyword_count])

In [23]:
# Apply calculation
full_df[['bias_wer', 'bias_wer_count']] = full_df.apply(calculate_bias_wer, axis=1)

# Show results
print(full_df[['audio', 'provider', 'bias_wer', 'bias_wer_count']].head(5))

# Calculate average Bias WER per provider
print("\nAverage Bias WER per provider:")
print(full_df.groupby('provider')['bias_wer'].mean())
print("\nTotal Keywords used for calculation per provider:")
print(full_df.groupby('provider')['bias_wer_count'].sum())

   audio provider  bias_wer  bias_wer_count
0      1  whisper     0.125             8.0
1      1  whisper     0.125             8.0
2      1  whisper     0.125             8.0
3      1  whisper     0.125             8.0
4      1  whisper     0.500             8.0

Average Bias WER per provider:
provider
amazon     0.119230
azure      0.106694
google     0.066970
whisper    0.095074
Name: bias_wer, dtype: float64

Total Keywords used for calculation per provider:
provider
amazon     7100.0
azure      7100.0
google     7100.0
whisper    7100.0
Name: bias_wer_count, dtype: float64


In [24]:
# Calculate WER grouped by keyword type (key)
keyword_stats = []

for index, row in full_df.iterrows():
    audio_id = str(row['audio'])
    provider = row['provider']
    
    # Get reference
    reference = ground_truth_map.get(audio_id, "")
    if not reference:
        continue
        
    hypothesis = str(row['text_normalized'])
    
    # Get keywords for this audio
    if audio_id not in keywords_data:
        continue
        
    # Map keyword values to their types (keys)
    # Note: A value might appear multiple times, but we'll use a simple map for now
    # Assuming values are unique enough within an audio or map to the same key type
    val_to_key = {k['val']: k['key'] for k in keywords_data[audio_id]['keywords']}
    target_keywords = set(val_to_key.keys())
    
    if not target_keywords:
        continue
        
    # Use the shared function to get detailed stats
    detailed_stats = get_keyword_stats(reference, hypothesis, target_keywords, val_to_key)
    
    # Add provider info to stats
    for stat in detailed_stats:
        stat['provider'] = provider
        keyword_stats.append(stat)

# Create DataFrame from stats
stats_df = pd.DataFrame(keyword_stats)

# Group by provider and key_type
grouped_stats = stats_df.groupby(['provider', 'key_type']).sum().reset_index()
grouped_stats['wer'] = grouped_stats['error'] / grouped_stats['count']

# Pivot for better readability
pivot_table = grouped_stats.pivot(index='key_type', columns='provider', values='wer')

print("\nBias WER grouped by Keyword Type:")
print(pivot_table)

# Optional: Total count per key type to see sample size
count_pivot = grouped_stats.pivot(index='key_type', columns='provider', values='count')
print("\nSample counts per Keyword Type (per provider):")
print(count_pivot)

# Calculate total words per key_type (summing across providers, or just taking one if they are equal)
# Since the dataset is balanced (same audios per provider), the count should be the same for all providers.
# We can just take the count from one provider or the mean to be safe.
total_words_per_key = grouped_stats.groupby('key_type')['count'].mean()
print("\nTotal words per key_type (per provider):")
print(total_words_per_key)


Bias WER grouped by Keyword Type:
provider     amazon     azure    google   whisper
key_type                                         
atributo   0.158000  0.233333  0.102667  0.156667
cantidad   0.086154  0.024615  0.020769  0.010769
cliente    0.240625  0.176875  0.139375  0.200000
codigo     0.185714  0.095714  0.064286  0.075714
marca      0.300000  0.550000  0.460000  0.290000
producto   0.072222  0.083333  0.046667  0.045556
proveedor  0.103333  0.080000  0.020000  0.070000
tiempo     0.037500  0.035000  0.010000  0.030000
ubicacion  0.220000  0.040000  0.050000  0.160000
vendedor   0.045000  0.015000  0.030000  0.035000

Sample counts per Keyword Type (per provider):
provider   amazon  azure  google  whisper
key_type                                 
atributo     1500   1500    1500     1500
cantidad     1300   1300    1300     1300
cliente      1600   1600    1600     1600
codigo        700    700     700      700
marca         100    100     100      100
producto      900    90